# Módulos estándar · Más módulos

`enum`, `pathlib`, `argparse`, `json`, `logging`, `shutil`, `tempfile`, `textwrap`, `string.Template` y `statistics`.

## 1. `Enum` y `auto`

Crear el `Enum` `Palo` de la baraja española con valores `auto()`, recorrerlo y acceder a sus miembros por nombre y por valor.

In [1]:
from enum import Enum, Flag, IntEnum, auto

class Palo(Enum):
    OROS = auto()
    COPAS = auto()
    ESPADAS = auto()
    BASTOS = auto()

for palo in Palo:
    print(palo, palo.name, palo.value)

assert Palo["COPAS"] is Palo(2) is Palo.COPAS
assert Palo.OROS != 1
try:
    Palo(9)
except ValueError as e:
    print(e)

Palo.OROS OROS 1
Palo.COPAS COPAS 2
Palo.ESPADAS ESPADAS 3
Palo.BASTOS BASTOS 4
9 is not a valid Palo


## 2. `IntEnum` y `Flag`

Crear un `IntEnum` `Prioridad` que se compare y ordene como un entero, y un `Flag` `Permiso` que combine lectura, escritura y ejecución con `|`.

In [2]:
class Prioridad(IntEnum):
    BAJA = 1
    MEDIA = 2
    ALTA = 3

tareas = [("documentar", Prioridad.BAJA), ("corregir fallo", Prioridad.ALTA), ("revisar", Prioridad.MEDIA)]
for tarea, prioridad in sorted(tareas, key=lambda t: t[1], reverse=True):
    print(f"{prioridad.name:<5} {tarea}")
assert Prioridad.ALTA > 2 and Prioridad(1) is Prioridad.BAJA

class Permiso(Flag):
    LEER = auto()
    ESCRIBIR = auto()
    EJECUTAR = auto()

usuario = Permiso.LEER | Permiso.ESCRIBIR
print(usuario, usuario.value, Permiso.ESCRIBIR in usuario, Permiso.EJECUTAR in usuario)
print([p.name for p in usuario])
todos = Permiso.LEER | Permiso.ESCRIBIR | Permiso.EJECUTAR
assert todos & ~Permiso.EJECUTAR == usuario

ALTA  corregir fallo
MEDIA revisar
BAJA  documentar
Permiso.LEER|ESCRIBIR 3 True False
['LEER', 'ESCRIBIR']


## 3. `pathlib`

Crear una carpeta de pruebas con subcarpetas y ficheros, recorrerla con `rglob`, contar los ficheros por extensión y renombrar los `.txt` a `.md` con un prefijo numérico.

In [3]:
import shutil
from collections import Counter
from pathlib import Path

PRUEBAS = Path("pruebas_modulos")
shutil.rmtree(PRUEBAS, ignore_errors=True)
for nombre in ["notas/tema1.txt", "notas/tema2.txt", "notas/resumen.md", "datos/ventas.csv",
               "datos/clientes.csv", "datos/2026/enero.csv", "leeme.txt"]:
    ruta = PRUEBAS / nombre
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(f"Contenido de {ruta.name}", encoding="utf-8")

ficheros = sorted(p for p in PRUEBAS.rglob("*") if p.is_file())
for p in ficheros:
    print(p.relative_to(PRUEBAS).as_posix(), p.stat().st_size, "bytes")
print(Counter(p.suffix for p in ficheros))

datos/2026/enero.csv 22 bytes
datos/clientes.csv 25 bytes
datos/ventas.csv 23 bytes
leeme.txt 22 bytes
notas/resumen.md 23 bytes
notas/tema1.txt 22 bytes
notas/tema2.txt 22 bytes
Counter({'.csv': 3, '.txt': 3, '.md': 1})


In [4]:
for i, p in enumerate(sorted(PRUEBAS.rglob("*.txt")), 1):
    p.rename(p.with_name(f"{i:02}-{p.stem}.md"))

print(sorted(p.relative_to(PRUEBAS).as_posix() for p in PRUEBAS.rglob("*.md")))
assert not list(PRUEBAS.rglob("*.txt")) and len(list(PRUEBAS.rglob("*.md"))) == 4

['01-leeme.md', 'notas/02-tema1.md', 'notas/03-tema2.md', 'notas/resumen.md']


## 4. `argparse`

Crear un analizador para un programa `copiar` con un origen, uno o varios destinos, `--veces` entero, `--modo` con opciones fijas y `--verbose`, y probarlo pasando listas de argumentos a `parse_args`, también con argumentos erróneos.

In [5]:
import argparse
import contextlib
import io

parser = argparse.ArgumentParser(prog="copiar", description="Copia un fichero en varios destinos.")
parser.add_argument("origen")
parser.add_argument("destinos", nargs="+")
parser.add_argument("-n", "--veces", type=int, default=1)
parser.add_argument("--modo", choices=["rapido", "seguro"], default="seguro")
parser.add_argument("-v", "--verbose", action="store_true")

args = parser.parse_args(["datos.csv", "a/", "b/", "-n", "3", "--verbose"])
print(args)
assert args.destinos == ["a/", "b/"] and args.veces == 3 and args.modo == "seguro" and args.verbose

def probar(argumentos):
    errores = io.StringIO()
    try:
        with contextlib.redirect_stderr(errores):
            return parser.parse_args(argumentos)
    except SystemExit:
        print("Error:", errores.getvalue().strip().splitlines()[-1])

probar(["datos.csv"])
probar(["datos.csv", "a/", "--veces", "tres"])
probar(["datos.csv", "a/", "--modo", "lento"])
print()
parser.print_help()

Namespace(origen='datos.csv', destinos=['a/', 'b/'], veces=3, modo='seguro', verbose=True)
Error: copiar: error: the following arguments are required: destinos
Error: copiar: error: argument -n/--veces: invalid int value: 'tres'
Error: copiar: error: argument --modo: invalid choice: 'lento' (choose from 'rapido', 'seguro')

usage: copiar [-h] [-n VECES] [--modo {rapido,seguro}] [-v]
              origen destinos [destinos ...]

Copia un fichero en varios destinos.

positional arguments:
  origen
  destinos

options:
  -h, --help            show this help message and exit
  -n, --veces VECES
  --modo {rapido,seguro}
  -v, --verbose


## 5. Subcomandos

Crear un programa `calc` con los subcomandos `sumar` (varios números) y `saludar` (un nombre y `--veces`), asociando a cada uno su función con `set_defaults`.

In [6]:
def sumar(args):
    return sum(args.numeros)

def saludar(args):
    return " ".join([f"¡Hola, {args.nombre}!"] * args.veces)

calc = argparse.ArgumentParser(prog="calc")
subcomandos = calc.add_subparsers(dest="orden", required=True)

p_sumar = subcomandos.add_parser("sumar", help="suma números")
p_sumar.add_argument("numeros", nargs="+", type=float)
p_sumar.set_defaults(funcion=sumar)

p_saludar = subcomandos.add_parser("saludar", help="saluda a alguien")
p_saludar.add_argument("nombre")
p_saludar.add_argument("--veces", type=int, default=1)
p_saludar.set_defaults(funcion=saludar)

for linea in ["sumar 1 2 3.5", "saludar Ana --veces 2"]:
    args = calc.parse_args(linea.split())
    print(f"{linea!r} -> {args.orden}: {args.funcion(args)}")

args = calc.parse_args(["sumar", "4", "5"])
assert args.funcion(args) == 9

'sumar 1 2 3.5' -> sumar: 6.5
'saludar Ana --veces 2' -> saludar: ¡Hola, Ana! ¡Hola, Ana!


## 6. `json` con objetos propios

Serializar una lista de `Alumno` (una *dataclass* con una fecha) pasando `default` a `json.dumps`, y reconstruir los objetos con `object_hook` en `json.loads`.

In [7]:
import json
from dataclasses import asdict, dataclass
from datetime import date

@dataclass
class Alumno:
    nombre: str
    nacimiento: date
    notas: list

def a_json(obj):
    if isinstance(obj, Alumno):
        return {"__tipo__": "Alumno", **asdict(obj)}
    if isinstance(obj, date):
        return {"__tipo__": "date", "iso": obj.isoformat()}
    raise TypeError(f"{type(obj).__name__} no es serializable")

def desde_json(d):
    match d.pop("__tipo__", None):
        case "date":
            return date.fromisoformat(d["iso"])
        case "Alumno":
            return Alumno(**d)
    return d

alumnos = [Alumno("Ana", date(2008, 3, 14), [7.5, 9]), Alumno("Luis", date(2007, 11, 2), [5, 6.25])]
texto = json.dumps(alumnos, default=a_json, ensure_ascii=False)
print(texto)
recuperados = json.loads(texto, object_hook=desde_json)
print(recuperados)
assert recuperados == alumnos

try:
    json.dumps({1, 2}, default=a_json)
except TypeError as e:
    print(e)

[{"__tipo__": "Alumno", "nombre": "Ana", "nacimiento": {"__tipo__": "date", "iso": "2008-03-14"}, "notas": [7.5, 9]}, {"__tipo__": "Alumno", "nombre": "Luis", "nacimiento": {"__tipo__": "date", "iso": "2007-11-02"}, "notas": [5, 6.25]}]
[Alumno(nombre='Ana', nacimiento=datetime.date(2008, 3, 14), notas=[7.5, 9]), Alumno(nombre='Luis', nacimiento=datetime.date(2007, 11, 2), notas=[5, 6.25])]
set no es serializable


## 7. `logging`: niveles y formato

Crear un logger propio que escriba en un `StringIO` con el formato `nivel | nombre | mensaje`, comprobar que se descartan los mensajes por debajo del nivel elegido y cambiar el nivel sobre la marcha.

In [8]:
import logging

memoria = io.StringIO()
manejador = logging.StreamHandler(memoria)
manejador.setFormatter(logging.Formatter("%(levelname)-8s | %(name)s | %(message)s"))

log = logging.getLogger("tienda")
log.handlers.clear()
log.addHandler(manejador)
log.setLevel(logging.INFO)
log.propagate = False

log.debug("Carrito: %s", ["pan", "leche"])
log.info("Pedido %d creado", 1042)
log.warning("Stock bajo de %s", "leche")
log.setLevel(logging.DEBUG)
log.debug("Ahora sí se ve el detalle")
try:
    1 / 0
except ZeroDivisionError as e:
    log.error("Fallo al calcular el descuento: %s", e)

salida = memoria.getvalue()
print(salida, end="")
assert "Carrito" not in salida and "Ahora sí" in salida and "ERROR" in salida

INFO     | tienda | Pedido 1042 creado
WARNING  | tienda | Stock bajo de leche
DEBUG    | tienda | Ahora sí se ve el detalle
ERROR    | tienda | Fallo al calcular el descuento: division by zero


## 8. `logging` a fichero

Añadir al logger un `FileHandler` que guarde en `pruebas_modulos/tienda.log` solo los avisos y errores, con fecha y hora, mientras el `StringIO` sigue recibiéndolo todo.

In [9]:
fichero = logging.FileHandler(PRUEBAS / "tienda.log", mode="w", encoding="utf-8")
fichero.setLevel(logging.WARNING)
fichero.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
log.addHandler(fichero)

log.info("Pedido %d enviado", 1042)
log.error("Pago rechazado en el pedido %d", 1043)
log.removeHandler(fichero)
fichero.close()

lineas = (PRUEBAS / "tienda.log").read_text(encoding="utf-8").splitlines()
print(len(lineas), "línea(s) en el fichero, sin la fecha:", [linea.split(" ", 2)[2] for linea in lineas])
assert len(lineas) == 1 and lineas[0].endswith("ERROR Pago rechazado en el pedido 1043")
assert memoria.getvalue().count("1042") == 2

1 línea(s) en el fichero, sin la fecha: ['ERROR Pago rechazado en el pedido 1043']


## 9. `shutil` y `tempfile`

En una carpeta de `tempfile.TemporaryDirectory`, copiar la carpeta de pruebas sin el `.log` con `copytree`, comprimirla con `make_archive` y listar el ZIP; al terminar, borrar la carpeta de pruebas con `rmtree`.

In [10]:
import tempfile
import zipfile

with tempfile.TemporaryDirectory() as directorio:
    tmp = Path(directorio)
    copia = shutil.copytree(PRUEBAS, tmp / "copia", ignore=shutil.ignore_patterns("*.log"))
    archivo = shutil.make_archive(str(tmp / "copia"), "zip", root_dir=copia)
    with zipfile.ZipFile(archivo) as z:
        nombres = sorted(n for n in z.namelist() if not n.endswith("/"))
    print(Path(archivo).name, nombres)
    assert len(nombres) == 7

shutil.rmtree(PRUEBAS)
assert not tmp.exists() and not PRUEBAS.exists()
print("Carpetas temporal y de pruebas borradas")

copia.zip ['01-leeme.md', 'datos/2026/enero.csv', 'datos/clientes.csv', 'datos/ventas.csv', 'notas/02-tema1.md', 'notas/03-tema2.md', 'notas/resumen.md']
Carpetas temporal y de pruebas borradas


## 10. `textwrap`

Ajustar un párrafo a 40 columnas con `fill`, sangrarlo con `indent`, resumirlo con `shorten` y quitar la sangría común de un bloque con `dedent`.

In [11]:
import textwrap

parrafo = ("El módulo textwrap ajusta párrafos a un ancho fijo, algo útil para mostrar ayudas en la terminal, "
           "generar correos de texto plano o dar formato a informes sin depender de librerías externas.")

ajustado = textwrap.fill(parrafo, width=40)
print(ajustado, end="\n\n")
assert all(len(linea) <= 40 for linea in ajustado.splitlines())

print(textwrap.indent(ajustado, "> "), end="\n\n")
print(textwrap.shorten(parrafo, width=50, placeholder=" [...]"), end="\n\n")

bloque = """
    def saludo():
        return "hola"
"""
print(textwrap.dedent(bloque).strip())
assert textwrap.dedent(bloque).startswith("\ndef saludo")

El módulo textwrap ajusta párrafos a un
ancho fijo, algo útil para mostrar
ayudas en la terminal, generar correos
de texto plano o dar formato a informes
sin depender de librerías externas.

> El módulo textwrap ajusta párrafos a un
> ancho fijo, algo útil para mostrar
> ayudas en la terminal, generar correos
> de texto plano o dar formato a informes
> sin depender de librerías externas.

El módulo textwrap ajusta párrafos a un [...]

def saludo():
    return "hola"


## 11. `string.Template`

Generar avisos personalizados a partir de una plantilla con `$variables`, viendo la diferencia entre `substitute` (exige todas las claves) y `safe_substitute`.

In [12]:
from string import Template

aviso = Template("Hola, $nombre: tu pedido $pedido llegará el ${dia}. Total: $total €.")
clientes = [{"nombre": "Ana", "pedido": 1042, "dia": "lunes", "total": "35,90"},
            {"nombre": "Luis", "pedido": 1043, "dia": "martes", "total": "12,00"}]
for cliente in clientes:
    print(aviso.substitute(cliente))

incompleto = {"nombre": "Marta", "pedido": 1044}
try:
    aviso.substitute(incompleto)
except KeyError as e:
    print("Falta la clave", e)
print(aviso.safe_substitute(incompleto))
print(aviso.get_identifiers())
assert aviso.get_identifiers() == ["nombre", "pedido", "dia", "total"]

Hola, Ana: tu pedido 1042 llegará el lunes. Total: 35,90 €.
Hola, Luis: tu pedido 1043 llegará el martes. Total: 12,00 €.
Falta la clave 'dia'
Hola, Marta: tu pedido 1044 llegará el ${dia}. Total: $total €.
['nombre', 'pedido', 'dia', 'total']


## 12. `statistics`

Calcular la media, mediana, moda, desviación típica, cuartiles, correlación y recta de regresión entre horas de estudio y notas con `statistics`, comprobando la media, la varianza y la pendiente con sus fórmulas.

In [13]:
import statistics as st
from math import isclose

horas = [2, 3, 5, 1, 4, 6, 3, 8, 7, 5]
notas = [5.0, 5.5, 7.0, 3.5, 6.5, 8.0, 6.0, 9.5, 8.5, 7.0]

print("media:", st.mean(notas), "· mediana:", st.median(notas), "· modas:", st.multimode(notas))
print(f"desviación: {st.stdev(notas):.3f} (muestral), {st.pstdev(notas):.3f} (poblacional)")
print("cuartiles:", st.quantiles(notas, n=4))
print(f"correlación horas-nota: {st.correlation(horas, notas):.3f}")
pendiente, ordenada = st.linear_regression(horas, notas)
print(f"nota ≈ {pendiente:.3f} · horas + {ordenada:.3f}")

n = len(notas)
media = sum(notas) / n
assert isclose(st.mean(notas), media)
assert isclose(st.variance(notas), sum((x - media) ** 2 for x in notas) / (n - 1))
media_h = sum(horas) / n
covarianza = sum((h - media_h) * (y - media) for h, y in zip(horas, notas))
assert isclose(pendiente, covarianza / sum((h - media_h) ** 2 for h in horas))

media: 6.65 · mediana: 6.75 · modas: [7.0]
desviación: 1.765 (muestral), 1.674 (poblacional)
cuartiles: [5.375, 6.75, 8.125]
correlación horas-nota: 0.989
nota ≈ 0.786 · horas + 3.191
